In [ ]:
%pip install 'stable-baselines3[extra]'
%pip install sb3-contrib

In [ ]:
import os
import time
from dataclasses import dataclass
import numpy as np
import matplotlib.pyplot as plt

import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import BaseCallback

from blackjack_env.wrappers import ShiftWrapper, SafeV1ActionWrapper

log_dir = os.path.abspath(f"../results/sb3_blackjack_{int(time.time())}")
os.makedirs(log_dir, exist_ok=True)


def make_env(log_dir: str, seed: int) -> gym.Env:
    env = gym.make("Blackjack4game-v1", render_mode=None)
    env = ShiftWrapper(env)
    env = SafeV1ActionWrapper(env)
    env = Monitor(env, filename=os.path.join(log_dir, "monitor.csv"))
    env.reset(seed=seed)
    return env


def make_evaluation_env(seed: int) -> gym.Env:
    env = gym.make("Blackjack4game-v1", render_mode=None)
    env = ShiftWrapper(env)
    env = SafeV1ActionWrapper(env)
    env.reset(seed=seed)
    return env


def test_env_PPO(
    policy: str,
    env: gym.Env,
    verbose: int = 1,
    tensorboard_log: str = log_dir,
    **ppo_kwargs: dict,
) -> None:
    model = PPO(
        policy, env, verbose=verbose, tensorboard_log=tensorboard_log, **ppo_kwargs
    )
    model.learn(total_timesteps=50_000)
